# Task #44 — Tính đặc trưng mới và one-hot vùng (Story #8, giai đoạn 1)

Dựng dataframe đặc trưng đầy đủ theo danh sách đã chốt ở `docs/feature-list.md` (Task #40): join `items_total_weight_g`, tính `approval_gap_hours`/`estimated_delivery_days`/`order_purchase_month`, one-hot `customer_state`+`primary_seller_state` (Phương án A đã chốt ở Task #39). Chưa xử lý giá trị thiếu, chưa lưu file — để dành cho Task #45.

Lưu ý: giữ nguyên `is_delayed` 3 trạng thái (True/False/NA) — quyết định loại NA (Phương án A) thuộc về giai đoạn 2 (chia train/test), không làm ở đây.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/orders_labeled.csv", low_memory=False)

date_cols = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

bool_cols = [
    "payment_has_boleto", "payment_has_credit_card", "payment_has_debit_card",
    "payment_has_not_defined", "payment_has_voucher", "items_multi_seller",
]
for col in bool_cols:
    df[col] = df[col].astype("boolean")

df["is_delayed"] = df["is_delayed"].astype("boolean")

df.shape

(99441, 42)

## 1. `items_total_weight_g` — join `order_items` + `products`

Giống pattern đã dùng ở Task #38 (`notebooks/14_explore_candidate_features.ipynb`).

In [2]:
items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")

items_weight = items.merge(products[["product_id", "product_weight_g"]], on="product_id", how="left")
order_weight = items_weight.groupby("order_id")["product_weight_g"].sum(min_count=1).rename("items_total_weight_g")

df = df.merge(order_weight, left_on="order_id", right_index=True, how="left")
print("Thiếu items_total_weight_g:", df["items_total_weight_g"].isna().sum(), "/", len(df))

Thiếu items_total_weight_g: 791 / 99441


## 2. `approval_gap_hours`, `estimated_delivery_days`, `order_purchase_month`

Pattern đã dùng ở Task #38/#40 (`notebooks/14_explore_candidate_features.ipynb`).

In [3]:
df["approval_gap_hours"] = (df["order_approved_at"] - df["order_purchase_timestamp"]).dt.total_seconds() / 3600
df["estimated_delivery_days"] = (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / 86400
df["order_purchase_month"] = df["order_purchase_timestamp"].dt.month

print("Thiếu approval_gap_hours:", df["approval_gap_hours"].isna().sum())
print("Thiếu estimated_delivery_days:", df["estimated_delivery_days"].isna().sum())
print("Thiếu order_purchase_month:", df["order_purchase_month"].isna().sum())

Thiếu approval_gap_hours: 160
Thiếu estimated_delivery_days: 0
Thiếu order_purchase_month: 0


## 3. One-hot `customer_state` + `primary_seller_state`

Phương án A đã chốt ở Task #39 (`notebooks/15_region_pair_encoding_options.ipynb`) — 2 cột riêng, không ghép cặp.

In [4]:
df_onehot = pd.get_dummies(
    df, columns=["customer_state", "primary_seller_state"],
    prefix=["customer_state", "primary_seller_state"],
)

customer_state_cols = [c for c in df_onehot.columns if c.startswith("customer_state_")]
seller_state_cols = [c for c in df_onehot.columns if c.startswith("primary_seller_state_")]
print("Số cột one-hot customer_state:", len(customer_state_cols))
print("Số cột one-hot primary_seller_state:", len(seller_state_cols))

Số cột one-hot customer_state: 27
Số cột one-hot primary_seller_state: 23


## 4. Chọn tập đặc trưng cuối cùng theo `docs/feature-list.md`

Giữ `order_id` (định danh) và `is_delayed` (nhãn mục tiêu) kèm theo, không phải đặc trưng. Loại toàn bộ cột rò rỉ dữ liệu và cột chưa đưa vào danh sách (zip/city/seller_id, review_*, order_status, timestamp thô).

In [5]:
numeric_features = [
    "items_num_items", "items_num_products", "items_num_sellers",
    "items_total_price", "items_total_freight", "items_num_categories",
    "items_total_weight_g",
    "payment_total_value", "payment_num_rows", "payment_num_types", "payment_max_installments",
    "payment_value_boleto", "payment_value_credit_card", "payment_value_debit_card",
    "payment_value_not_defined", "payment_value_voucher",
    "approval_gap_hours", "estimated_delivery_days", "order_purchase_month",
]

boolean_features = [
    "payment_has_boleto", "payment_has_credit_card", "payment_has_debit_card",
    "payment_has_not_defined", "payment_has_voucher", "items_multi_seller",
]

region_onehot_features = customer_state_cols + seller_state_cols

final_columns = ["order_id"] + numeric_features + boolean_features + region_onehot_features + ["is_delayed"]

features_df = df_onehot[final_columns].copy()
print(features_df.shape)
features_df.dtypes.value_counts()

(99441, 77)


bool       50
float64    18
boolean     7
str         1
int32       1
Name: count, dtype: int64